# Elephant density and vegetation contrasts

The archived country summaries and patch observations are non-raster inputs. This notebook computes cached results and exports the country and site panels.

In [ ]:
import sys
from pathlib import Path

def find_repo(start):
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / 'config' / 'analysis.json').is_file():
            return candidate
    raise FileNotFoundError('Could not find config/analysis.json above the notebook directory')

ROOT = find_repo(Path.cwd())
sys.path.insert(0, str(ROOT / 'src'))
from elephant_analysis.runtime import configure_runtime
configure_runtime()

import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import gaussian_kde
from elephant_analysis.artifacts import load_arrays, save_arrays
from elephant_analysis.data_inputs import load_inputs
from elephant_analysis.density import compute_density_results

DATA_ROOT = ROOT / 'data'
RESULTS_ROOT = ROOT / 'results'
FIGURES_ROOT = ROOT / 'figures'
DENSITY_INPUT_MODE = 'reference'  # The recovered density tables cannot be rebuilt from TIFF alone.
RUN_ANALYSIS = True
%matplotlib inline
%config InlineBackend.figure_format = 'retina'

In [ ]:
density_result_path = RESULTS_ROOT / 'fig2_density_results.npz'

if RUN_ANALYSIS:
    density_input = load_inputs('density', DATA_ROOT, mode=DENSITY_INPUT_MODE)
    density_results = compute_density_results(density_input, n_perm=5000, seed=42)
    save_arrays(density_result_path, **density_results)

plot_data = load_arrays(density_result_path)
print('country rows:', len(plot_data['country_th']))
print('patch rows:', len(plot_data['patch_rows']))
print('country fit stats [slope, R2, p_perm, p_mean]:', plot_data['country_th_stats'], plot_data['country_tc_stats'])
print('patch fit stats [slope, R2, p_perm, p_mean]:', plot_data['patch_th_stats'], plot_data['patch_tc_stats'])

In [ ]:
# Original plotting cells copied from Main_figure.ipynb.
# Keep source sizes, font settings, labels, colours, positions and line widths.
EXPORT_PDF = True
EXPORT_DPI = 600
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from scipy import stats
from scipy.stats import kstest, gaussian_kde
from elephant_analysis import array_utils as gc
mpl.rcdefaults()
mpl.rcParams["pdf.fonttype"] = 42

def export_original(figure, stem, dpi=EXPORT_DPI):
    # Match the source savefig contract, including tight bounding boxes.
    FIGURES_ROOT.mkdir(parents=True, exist_ok=True)
    figure.savefig(FIGURES_ROOT / (stem + '.png'), dpi=dpi, bbox_inches='tight')
    if EXPORT_PDF:
        figure.savefig(FIGURES_ROOT / (stem + '.pdf'), bbox_inches='tight')

dens = plot_data['country_density']
val = plot_data['country_mean']
std = plot_data['country_se']
dens_site = pd.DataFrame({'TH': plot_data['country_th'], 'TC': plot_data['country_tc']})
# BEGIN ORIGINAL CELL 12
import matplotlib.pyplot as plt
from sklearn.metrics import r2_score
from scipy.stats import linregress
from sklearn.linear_model import TheilSenRegressor
import pingouin as pg
import statsmodels.api as sm
from scipy.stats import gaussian_kde
import warnings
from scipy.stats import wilcoxon
def annotate_wilcoxon_mean_p(ax, y, color, pos=(0.85, 0.85), fontsize=14):

    y = np.asarray(y, float)
    y = y[np.isfinite(y)]

    y_nonzero = y[y != 0]
    if y_nonzero.size < 5:
        return
    try:
        stat, p = wilcoxon(y_nonzero, alternative='two-sided',
                           zero_method='wilcox', mode='auto')
    except Exception:
        return
    if p < 1e-3:
        txt = r'$p_{\mathrm{mean}}<0.001$'
    elif p < 1e-2:
        txt = r'$p_{\mathrm{mean}}<0.01$'
    elif p < 5e-2:
        txt = r'$p_{\mathrm{mean}}<0.05$'
    else:
        txt = rf'$p_{{\mathrm{{mean}}}}={p:.2f}$'

    ax.text(pos[0], pos[1], txt,
            transform=ax.transAxes, ha='right', va='top',
            fontsize=14, color=color, zorder=10)















def linear_fit_ci_on_binned(ax, xm, ym, color, label=None, count=None,use_pred_interval=False, n_perm=5000, seed=42):

    prefix = 'country_' + ('th', 'tc')[i]
    cached_stats = plot_data[prefix + '_stats']
    p_slope = float(cached_stats[2])
    if p_slope >= 0.05:
        return float(cached_stats[1]), p_slope
    xx = plot_data[prefix + '_fit_x']
    y_pred = plot_data[prefix + '_fit_y']
    ci_low = plot_data[prefix + '_fit_ci_low']
    ci_high = plot_data[prefix + '_fit_ci_high']
    ax.plot(xx, y_pred, color=color, lw=2.5, label=label)
    ax.fill_between(xx, ci_low, ci_high, color=color, edgecolor="none", alpha=0.2)

    if p_slope < 1e-3:
        txt = r'$p_{\mathrm{fit}}<0.001$'
    elif p_slope < 1e-2:
        txt = r'$p_{\mathrm{fit}}<0.01$'
    elif p_slope < 5e-2:
        txt = r'$p_{\mathrm{fit}}<0.05$'
    else:
        txt = rf'$p_{{\mathrm{{mean}}}}={p_slope:.2f}$'

    ax.text(0.85, 0.92, txt,transform=ax.transAxes,ha='right', va='top',fontsize=14, color=color)
    return float(cached_stats[1]), p_slope

def draw_side_densities(ax, y_groups, colors, x_pos='right', pad_frac=0.02,
                        width_frac=0.10, n_pts=300, lw=1.5,
                        show_ci=True, ci_kind='normal', ci_alpha=1.0,
                        ci_edgecolor=None, ci_edgewidth=1.0):

    xmin, xmax = ax.get_xlim()
    ymin, ymax = ax.get_ylim()
    xspan = max(1e-12, xmax - xmin)

    if x_pos == 'right':
        x_right = xmax - pad_frac * xspan
    else:
        x_right = float(x_pos)

    width = width_frac * xspan

    yy = np.linspace(ymin, ymax, int(max(100, n_pts)))
    for idx, yg in enumerate(y_groups):
        if yg is None:
            continue
        y = np.asarray(yg, float)
        y = y[np.isfinite(y)]
        if y.size < 3:
            continue
        c = colors[idx % len(colors)]

        kde = gaussian_kde(y)
        dens = kde(yy)
        dens_scaled = dens / dens.max() * width if dens.max() > 0 else np.zeros_like(dens)

        ax.fill_betweenx(yy, x_right - dens_scaled, x_right, color=c, alpha=0.20, lw=0, zorder=1)
        ax.plot(x_right - dens_scaled, yy, color=c, lw=lw, zorder=3)

        mu = float(np.nanmean(y))
        i_mu = int(np.argmin(np.abs(yy - mu)))
        local_w = dens_scaled[i_mu]
        ax.hlines(mu, x_right - local_w, x_right, linestyle='--', colors=c, lw=lw, zorder=4)

%matplotlib inline
%config InlineBackend.figure_format = 'retina'
fig = plt.figure(figsize=(12,4))
fig.subplots_adjust(wspace=0.25,hspace=0.02, left=None, bottom=None, right=None, top=None)
font = {'family': 'sans-serif',
        'sans-serif': 'Arial',
        'weight': 'normal',
        'size': 16}
plt.rc('font', **font)
cl1 = [(47/255,85/255,151/255),(22/255,142/255,88/255)]
cl2  = [(150/255,176/255,222/255), (188/255,222/255,207/255)]
x1=np.linspace(0,1.3,1000)
x2=np.linspace(0,1.3,1000)
zero=np.zeros((1000))

for i in range(2):
    ax=fig.add_subplot(1,2,i+1)

    ax.errorbar(dens[i],val[i],yerr=[std[i],std[i]],fmt='o',ecolor=cl2[i], mfc='white', mec=cl1[i], ms=6, lw=1, capsize=0, zorder=3)
    plt.plot(x2,zero,color='black',linestyle='--',linewidth=1.5)
    linear_fit_ci_on_binned(ax, dens[i],val[i], color=cl1[i])

    plt.xlabel('Elephant density (N km$^{-2}$)',fontsize=18,labelpad=10)
    ax.set_xlim(0,1.25)
    ax.set_xticks(np.arange(0,1.25,0.3))
    ax=plt.gca()
    ax.yaxis.set_ticks_position('left')
    ax.spines['left'].set_position(('data',-0.03))
    if i==0:
        ax.set_ylim(-5,15)
        ax.set_yticks(np.arange(-5,15.01,5))
        ax.set_ylabel(''r'$\Delta$Tree height'" (m)",fontsize=18,labelpad=10)
    if i==1:
        ax.set_ylim(-20,40)
        ax.set_yticks(np.arange(-20,40.01,20))
        ax.set_ylabel(''r'$\Delta$Tree cover'" (%)",fontsize=18,labelpad=10)
    ax.xaxis.label.set_size(18)
    ax.yaxis.label.set_size(18)
    ax.tick_params(labelsize=14)
    ax.spines['right'].set_visible(False)
    ax.spines['top'].set_visible(False)
    ax.xaxis.set_tick_params(width=2)
    ax.yaxis.set_tick_params(width=2)
    ax.spines['left'].set_linewidth(2)
    ax.spines['bottom'].set_linewidth(2)

    if i == 0:
        y_groups = [dens_site["TH"].values]
    elif i == 1:
        y_groups = [dens_site["TC"].values]
    draw_side_densities(ax,y_groups=y_groups,colors=[cl1[i]],x_pos='right', pad_frac=0.0,width_frac=0.08,n_pts=300,lw=2)
    if i == 0:
        y_for_test = dens_site["TH"].values
    else:
        y_for_test = dens_site["TC"].values
    annotate_wilcoxon_mean_p(ax, y_for_test, color=cl1[i], pos=(0.85, 0.85), fontsize=14)

warnings.filterwarnings("ignore", category=UserWarning, module="matplotlib.font_manager")
# END ORIGINAL CELL 12
export_original(fig, 'fig2a_b_country_density')
plt.show()

patches = pd.DataFrame(plot_data['patch_rows'], columns=plot_data['patch_columns'].astype(str))
del_th = patches['TH'].to_numpy()
del_tc = patches['TC'].to_numpy()
ele_dens = patches['Ele_Dens'].to_numpy()
# BEGIN ORIGINAL CELL 16
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as colors
from scipy.stats import gaussian_kde
import statsmodels.api as sm
from sklearn.metrics import r2_score
import warnings
%matplotlib inline
%config InlineBackend.figure_format = 'retina'

fig = plt.figure(figsize=(12, 4))
fig.subplots_adjust(wspace=0.3, hspace=0, left=None, bottom=None, right=None, top=None)
font = {'family': 'sans-serif',
        'sans-serif': 'Arial',
        'weight': 'normal',
        'size': 16}
plt.rc('font', **font)

th   = del_th
tc   = del_tc
dens = ele_dens













































def draw_side_densities(ax, y_groups, colors, x_pos='right', pad_frac=0.02,
                        width_frac=0.10, n_pts=300, lw=1.5):

    xmin, xmax = ax.get_xlim()
    ymin, ymax = ax.get_ylim()
    xspan = max(1e-12, xmax - xmin)

    if x_pos == 'right':
        x_right = xmax - pad_frac * xspan
    else:
        x_right = float(x_pos)

    width = width_frac * xspan

    yy = np.linspace(ymin, ymax, int(max(100, n_pts)))

    for idx, yg in enumerate(y_groups):
        if yg is None:
            continue
        y = np.asarray(yg, float)
        y = y[np.isfinite(y)]
        if y.size < 3:
            continue

        c = colors[idx % len(colors)]

        kde = gaussian_kde(y)
        dens = kde(yy)
        if dens.max() > 0:
            dens_scaled = dens / dens.max() * width
        else:
            dens_scaled = np.zeros_like(dens)

        ax.fill_betweenx(yy, x_right - dens_scaled, x_right,
                         color=c, alpha=0.20, lw=0, zorder=1)
        ax.plot(x_right - dens_scaled, yy, color=c, lw=lw, zorder=3)

        mu = float(np.nanmean(y))
        i_mu = np.argmin(np.abs(yy - mu))
        local_w = dens_scaled[i_mu]
        ax.hlines(mu, x_right - local_w, x_right,
                  linestyle='--', colors=c, lw=lw, zorder=4)
from scipy.stats import wilcoxon, linregress

def annotate_wilcoxon_mean_p(ax, y, color, pos=(0.85, 0.82), fontsize=14):

    y = np.asarray(y, float)
    y = y[np.isfinite(y)]

    y_nonzero = y[y != 0]
    if y_nonzero.size < 5:
        return

    try:
        stat, p = wilcoxon(y_nonzero, alternative='two-sided',
                           zero_method='wilcox', mode='auto')
    except Exception:
        return

    if p < 1e-3:
        txt = r'$p_{\mathrm{mean}}<0.001$'
    elif p < 1e-2:
        txt = r'$p_{\mathrm{mean}}<0.01$'
    elif p < 5e-2:
        txt = r'$p_{\mathrm{mean}}<0.05$'
    else:
        txt = rf'$p_{{\mathrm{{mean}}}}={p:.2f}$'

    ax.text(pos[0], pos[1], txt,transform=ax.transAxes, ha='right', va='top',
            fontsize=fontsize, color=color, zorder=10)























def truncate_colormap(cmap, minval=0.0, maxval=1.0, n=50):
    new_cmap = colors.LinearSegmentedColormap.from_list(
        "trunc({n},{a:.2f},{b:.2f})".format(n=cmap.name, a=minval, b=maxval),
        cmap(np.linspace(minval, maxval, n)))
    return new_cmap
trunc_cl1 = truncate_colormap(plt.get_cmap("Greys"), 0.1, 0.8)
trunc_cl2 = truncate_colormap(plt.get_cmap("Greys"), 0.1, 0.8)
cmaps = [trunc_cl1, trunc_cl2]

cl1  = [(47/255,85/255,151/255), (22/255,142/255,88/255)]
cl2  = [(150/255,176/255,222/255), (188/255,222/255,207/255)]

pairs = [(th, r'$\Delta$Tree height$\,(\mathrm{m})$'),
         (tc, r'$\Delta$Tree cover$\,(\%)$')]
im_last = None
for i, (y_raw, ylabel) in enumerate(pairs, start=1):
    ax = fig.add_subplot(1, 2, i)

    x_arr = np.asarray(dens)
    y_arr = np.asarray(y_raw)
    ok = np.isfinite(x_arr) & np.isfinite(y_arr)
    x_arr = x_arr[ok]
    y_arr = y_arr[ok]

    prefix = 'patch_' + ('th', 'tc')[i-1]
    xm = plot_data[prefix + '_bin_x']
    ym = plot_data[prefix + '_bin_y']
    se = plot_data[prefix + '_bin_se']

    xy = np.vstack([x_arr, y_arr])
    z = gaussian_kde(xy)(xy)
    idx = z.argsort()
    x_s, y_s, z_s = x_arr[idx], y_arr[idx], z[idx]
    z_stretch = z_s ** 0.6
    im = ax.scatter(x_s, y_s, c=z_s, cmap=cmaps[i-1],s=10)

    xmin, xmax = float(np.min(x_arr)), float(np.max(x_arr))
    xx = plot_data[prefix + '_fit_x']
    y_pred = plot_data[prefix + '_fit_y']
    ci_low = plot_data[prefix + '_fit_ci_low']
    ci_high = plot_data[prefix + '_fit_ci_high']
    r2 = float(plot_data[prefix + '_stats'][1])
    p_fit = float(plot_data[prefix + '_stats'][2])
    ax.plot(xx, y_pred, color=cl1[i-1], lw=2.5)
    ax.fill_between(xx, ci_low, ci_high,color=cl1[i-1], alpha=0.18, linewidth=0)

    if xm.size > 0:
        ax.errorbar(xm, ym, yerr=se,fmt='o', mfc='white',mec=cl1[i-1],ecolor=cl2[i-1],elinewidth=1,
                    capsize=0,ms=6,label='Bin mean ± SE')

    y_max, y_min = np.nanmax(y_arr), np.nanmin(y_arr)
    yr = max(1e-9, y_max - y_min)
    x_text = xmin + 0.70 * (xmax - xmin)
    y_text1 = y_max - 0.12 * yr
    y_text2 = y_max - 0.24 * yr

    if np.isfinite(p_fit):
        if p_fit < 1e-3:
            txt_fit = r'$p_{\mathrm{fit}}<0.001$'
        elif p_fit < 1e-2:
            txt_fit = r'$p_{\mathrm{fit}}<0.01$'
        elif p_fit < 5e-2:
            txt_fit = r'$p_{\mathrm{fit}}<0.05$'
        else:
            txt_fit = rf'$p_{{\mathrm{{fit}}}}={p_fit:.2f}$'

        ax.text(0.6, 0.9, txt_fit,fontsize=14, transform=ax.transAxes, color=cl1[i-1])

    ax.set_xlabel('Elephant density (N km$^{-2}$)', fontsize=18, labelpad=10)
    ax.set_ylabel(ylabel, fontsize=18, labelpad=10)
    ax.set_xlim(0, 3.8)
    ax.set_xticks(np.arange(0, 3.8, 1))
    xx = np.linspace(xmin, 3.8, 400)
    ax.plot(xx, np.zeros_like(xx), linestyle='--',linewidth=1.5, color='black')
    ax=plt.gca()
    ax.yaxis.set_ticks_position('left')
    ax.spines['left'].set_position(('data',-0.09))
    if i == 1:
        ax.set_ylim(-3, 6)
        ax.set_yticks(np.arange(-3, 6.01, 3))
    else:
        ax.set_ylim(-15, 20)
        ax.set_yticks(np.arange(-15, 20.01, 5))

    ax.tick_params(labelsize=14)
    ax.spines['right'].set_visible(False)
    ax.spines['top'].set_visible(False)
    ax.spines['left'].set_linewidth(2)
    ax.spines['bottom'].set_linewidth(2)
    ax.xaxis.set_tick_params(width=2)
    ax.yaxis.set_tick_params(width=2)

    draw_side_densities(ax,y_groups=[y_arr], colors=[cl1[i-1]],x_pos='right',pad_frac=0.0,width_frac=0.08,n_pts=300,lw=2)

    annotate_wilcoxon_mean_p(ax,y_arr,color=cl1[i-1],pos=(0.85, 0.87),fontsize=14)
warnings.filterwarnings("ignore", category=UserWarning, module="matplotlib.font_manager")
# END ORIGINAL CELL 16
export_original(fig, 'fig2c_d_patch_density')
plt.show()
